# Multiprocessing Genetic Algorithm

This notebook implements a multiprocessing version of the genetic algorithm from v2.ipynb.
Each CPU core runs its own independent GA population, and all DNA vectors and scores are saved.

In [1]:
# Import the multiprocessing GA module
import sys
sys.path.append('.')
import numba
import pickle
import numpy as np
import multiprocessing as mp
from pathlib import Path
import matplotlib.pyplot as plt
from v2_multiprocessing import run_multiprocess_ga, GA_CONFIG

In [2]:
# Show available configurations
print("Available GA configurations:")
for config_name, config in GA_CONFIG.items():
    print(f"{config_name:15s}: {config['NUM_GENERATIONS']:3d} gen, {config['POP_SIZE']:3d} pop, bounds {config['DNA_BOUNDS']}")

print(f"\nAvailable CPU cores: {mp.cpu_count()}")

Available GA configurations:
single         :   1 gen,   1 pop, bounds [0, 500]
small          :  10 gen, 100 pop, bounds [0, 500]
medium         :  20 gen, 100 pop, bounds [0, 500]
large          : 100 gen, 500 pop, bounds [0, 500]
E              : 300 gen, 300 pop, bounds [0, 500]
F              : 120 gen, 150 pop, bounds [0, 500]
G              : 250 gen, 1000 pop, bounds [0, 500]
H              : 300 gen, 1000 pop, bounds [0, 500]

Available CPU cores: 10


In [3]:
# Configure and run multiprocessing GA
config_name = "small"  # Change this to desired config
num_processes = 4      # Change this to desired number of cores (None = all cores)
num_generations = 5    # Change this to desired generations per core (None = from config)

print(f"Running GA with config '{config_name}' on {num_processes} processes...")
print(f"Each process will run {num_generations} generations")
print(f"Total individuals to test: {num_processes * GA_CONFIG[config_name]['POP_SIZE'] * num_generations:,}")
print("\nStarting multiprocessing GA...")

Running GA with config 'small' on 4 processes...
Each process will run 5 generations
Total individuals to test: 2,000

Starting multiprocessing GA...


In [ ]:
# Run the multiprocessing GA
results = run_multiprocess_ga(
    config_name=config_name,
    num_processes=num_processes, 
    num_generations=num_generations,
    results_dir=None  # Will create timestamped directory
)

In [ ]:
# Analyze results
summary = results['summary']
all_dna = results['all_dna_tested']
gen_stats = results['generation_stats']

print(f"Results Summary:")
print(f"  Total individuals tested: {summary['total_individuals_tested']:,}")
print(f"  Best overall score: {summary['best_overall_score']}")
print(f"  Runtime: {summary['total_runtime']:.2f} seconds")
print(f"  Performance: {summary['individuals_per_second']:.1f} individuals/second")
print(f"  Results directory: {summary['results_directory']}")

In [ ]:
# Plot fitness evolution across all processes
plt.figure(figsize=(12, 8))

# Plot best scores per generation for each process
for process_id in range(summary['successful_processes']):
    process_stats = [stat for stat in gen_stats if stat['process_id'] == process_id]
    generations = [stat['generation'] for stat in process_stats]
    best_scores = [stat['best_score'] for stat in process_stats]
    plt.plot(generations, best_scores, alpha=0.7, label=f'Process {process_id}')

plt.xlabel('Generation')
plt.ylabel('Best Fitness Score')
plt.title('Fitness Evolution Across All Processes')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze score distribution
all_scores = [dna['total_score'] for dna in all_dna]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(all_scores, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Fitness Score')
plt.ylabel('Frequency')
plt.title('Distribution of All Fitness Scores')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([all_scores])
plt.ylabel('Fitness Score')
plt.title('Fitness Score Box Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Score Statistics:")
print(f"  Min: {min(all_scores)}")
print(f"  Max: {max(all_scores)}")
print(f"  Mean: {np.mean(all_scores):.2f}")
print(f"  Std: {np.std(all_scores):.2f}")
print(f"  Median: {np.median(all_scores):.2f}")

In [ ]:
# Find and display best DNA vectors
best_dna = sorted(all_dna, key=lambda x: x['total_score'], reverse=True)[:10]

print("Top 10 DNA vectors and their scores:")
print("=" * 80)
for i, dna_record in enumerate(best_dna):
    print(f"Rank {i+1}:")
    print(f"  Total Score: {dna_record['total_score']} (Exp: {dna_record['exp_score']}, Cont: {dna_record['cont_score']})")
    print(f"  Process: {dna_record['process_id']}, Generation: {dna_record['generation']}, Individual: {dna_record['individual_id']}")
    print(f"  DNA: {dna_record['dna'][:10]}... (showing first 10 genes)")
    print()

In [ ]:
# Save best DNA vectors for further analysis
best_dna_file = Path(summary['results_directory']) / "best_dna_vectors.pkl"
with open(best_dna_file, 'wb') as f:
    pickle.dump(best_dna, f)

print(f"Saved top {len(best_dna)} DNA vectors to: {best_dna_file}")

# You can load them later with:
# with open(best_dna_file, 'rb') as f:
#     loaded_best_dna = pickle.load(f)

In [ ]:
# with open(best_dna_file, 'rb') as f:
#     loaded_best_dna = pickle.load(f)

In [ ]:
# Performance analysis across processes
process_performance = []
for result in results['process_results']:
    performance = {
        'process_id': result['process_id'],
        'individuals_tested': result['total_individuals_tested'],
        'best_score': result['best_overall_score'],
        'runtime': result['completion_time'] - summary['start_time']
    }
    performance['individuals_per_sec'] = performance['individuals_tested'] / performance['runtime']
    process_performance.append(performance)

print("Process Performance Summary:")
print("Process | Individuals | Best Score | Runtime (s) | Ind/Sec")
print("-" * 60)
for perf in process_performance:
    print(f"{perf['process_id']:7d} | {perf['individuals_tested']:11,d} | {perf['best_score']:10d} | {perf['runtime']:11.2f} | {perf['individuals_per_sec']:7.1f}")